# RAG-pipeline: fast kjede med kalkulator

RAG-pipelinen er en **fast** kjede. Den kjører de samme fem stegene i samme rekkefølge for hvert spørsmål, uten løkker og uten nye forsøk. Stegene følger RAG-pipelinen fra forelesning 4 (lysbilde «RAG-pipeline med kilder»), men generatoren skriver et *regnestykke* og ikke et ferdig svar:

| Fase (forel. 4) | Steg i RAG-pipelinen | Celle |
|---|---|---|
| Indekseringstid | Dokumentgrunnlag → chunking → indeks | 1–3 |
| Spørretid | 1. **Retrieval:** hent `TOP_K` chunks fra spørsmålets eget dokument | 4 |
| | 2. **Kontekstpakking:** én kontekst der hver chunk har sin chunk-ID | 5 |
| | 3. **Generering:** LLM-en skriver *ett* program i kalkulatorens format (ett kall, ingen retry) | 6 |
| | 4. **Beregning:** programmet kjøres med `src/kalkulator.py`, den samme kalkulatoren som agenten bruker | 7 |
| | 5. **Formatering og retting:** svaret sammenlignes med fasit med de felles reglene | 7 |

Forskjellen fra agenten er hvem som styrer: her bestemmer Python rekkefølgen, i agenten bestemmer modellen.

Hvert steg er et mulig **feilsted** (forel. 4, tabellen «Feilsteder i RAG-pipelinen»). Derfor logger vi mellomresultatene fra hvert steg, slik at en feil kan plasseres i riktig ledd.

**Felles kode.** Alt som skal være likt i RAG-pipelinen og agenten, ligger i `src/` og importeres her: innstillinger, utvalg, korpus og søk, kalkulator, prompt-regler, modellkall, retting, loggformat og evalueringstabeller. Det RAG-pipeline-spesifikke er kontekstpakkingen, at prompten ber om ett helt program, og den faste rekkefølgen.

**Designvalg.** Søkemetode, chunking, `top_k` og regelen mot nøsting er testet på valideringsutvalget i `pilot_val.ipynb`.

## 0. Oppsett

Alle innstillingene ligger i `src/config.py`, den samme filen agenten bruker. Det eneste som velges her, er **hvilket utvalg** vi kjører på.

**Lagrede resultater.** Notebooken gjør ingen API-kall. Kjøringen på hele testutvalget leses fra `results/`, og det er disse resultatene som står i rapporten. **Valgfritt ekstra steg:** seksjon 12 sist i notebooken kan kjøre RAG-pipelinen live mot OpenAI på 8 testspørsmål, hvis du vil se metoden i bruk med ekte API-kall (krever egen API-nøkkel).

**Demo (ca. 1–2 minutter, noen cent).** Kjør alle cellene. Med `LIVE_DEMO = True` kjøres hele RAG-pipelinen live mot OpenAI på de `N_DEMO` = 8 første testspørsmålene (seksjon 8). Svarene sammenlignes med den lagrede testkjøringen. Alt etter demoen, med evaluering på alle 200 testspørsmålene, leser den lagrede testkjøringen i `results/`. Uten API-nøkkel i `.env` hoppes live-cellene over, og resten av notebooken kjører som vanlig.

In [1]:
# --- Les inn endringer i src/ automatisk (ellers husker Python gamle versjoner av modulene) ---
%load_ext autoreload
%autoreload 2

import json
import re
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

# --- Finn prosjektroten og gjør src/ importerbar ---
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

# --- Felles innstillinger (samme fil som agenten bruker) ---
from src.config import MODEL, TOP_K, RESULTS_DIR, settings

# --- Felles byggeklosser ---
from src.utvalg import load_utvalg, load_examples                 # samme spørsmål som agenten
from src.korpus import Korpus, make_chunks, gold_coverage         # samme korpus og søk som agenten
from src.kalkulator import run_program                            # samme kalkulator (RAG-pipelinen bruker hele programmer)
from src.prompts import B2_SYSTEM_PROMPT, B2_EXAMPLE_USER, B2_EXAMPLE_ASSISTANT   # tallregler felles med agenten
from src.llm import call_model, usage                             # samme modellkall og tokentelling
from src.evaluering import (is_correct, is_correct_rounded, is_scale_error,
                            format_answer, classify_failure)      # samme retting som agenten
from src.logg import make_record, write_jsonl, read_jsonl, same_settings   # samme loggfelt som agenten
from src.rapport import oppsummering, failure_log, failure_oversikt   # samme evalueringstabeller

# --- Det eneste valget i denne notebooken: hvilket utvalg ---
# "dev": all utvikling og justering skjer her.
# "test": kjøres ÉN gang, når både RAG-pipelinen og agenten er ferdige. Ikke bytt før det!
SPLIT = "test"
CONDITION = "RAG-pipeline"    # betingelsen, står i hver loggrad

pd.set_option("display.max_colwidth", 120)

# --- Vis innstillingene ---
# settings() er det samme som logges i hver rad, så vi alltid vet hvilket oppsett et resultat kom fra.
{"condition": CONDITION, "split": SPLIT, **settings()}

{'condition': 'RAG-pipeline',
 'split': 'test',
 'model': 'gpt-4.1',
 'temperature': 0,
 'seed': 42,
 'retrieval_scope': 'dokument',
 'chunk_strategy': 'setning',
 'index_backend': 'embedding',
 'embedding_model': 'sentence-transformers/all-MiniLM-L6-v2',
 'top_k': 5,
 'max_agent_steps': 8,
 'rel_tol': 0.01}

## 1. Dokumentgrunnlag og utvalg

Forelesning 4 sier at *korpuset setter en øvre grense for hva systemet kan finne*. I FinQA er dokumentgrunnlaget for hvert spørsmål én rapportside:

- `pre_text`: tekst før tabellen
- `table`: selve tabellen
- `post_text`: tekst etter tabellen

Til hvert spørsmål finnes fasitsvaret `exe_ans`, fasitprogrammet `program` og `gold_inds`, som er de chunkene fasiten bygger på. `gold_inds` lar oss måle **retrieval-dekning**: finnes nødvendig evidens i top-k? (forel. 4, «Kildeforankring er en påstand som må testes»).

**Utvalget er felles med agenten** og trekkes én gang av `src/utvalg.py`:
- **Fasit:** `exe_ans`, tallet fasitprogrammet gir (`0.01714`), ikke `answer`, som er tekst skrevet av mennesker (`"1.7%"`).
- **Filter:** bare spørsmål der fasitprogrammet kun bruker add, subtract, multiply og divide. Ellers ville vi målt kalkulatorens begrensninger.
- **dev:** 50 spørsmål fra `dev.json`, seed 42. All justering skjer her.
- **val:** 100 spørsmål fra `train.json`, seed 42. Designvalgene testes her (`pilot_val.ipynb`).
- **test:** 200 spørsmål fra `test.json`, lagret som et fast utvalg i `data/utvalg_test.csv`. Kjøres én gang til slutt.

In [2]:
# --- Steg 1: Les utvalget (samme fil som agenten leser) ---
utvalg = load_utvalg(SPLIT)
examples = load_examples(SPLIT)     # id -> hele FinQA-eksempelet (tekst, tabell, qa)

print({"split": SPLIT, "spørsmål": len(utvalg)})

# --- Steg 2: Fordeling av regnesteg (avtalt inndeling: ett vs. flere steg) ---
display(utvalg["n_steps"].map(lambda n: "1 steg" if n == 1 else "2+ steg")
        .value_counts().rename("antall spørsmål").to_frame())

# De fem første spørsmålene i utvalget
utvalg.head()

{'split': 'test', 'spørsmål': 200}


,antall spørsmål
n_steps,
1 steg,110
2+ steg,90


,id,question,gold_answer,gold_program,n_steps,gold_inds
0,AMT/2016/page_49.pdf-4,"what is the average number of shares per registered holder as of february 17 , 2017?",2.792124e+06,"divide(427195037, 153)",1,[text_3]
1,ETR/2011/page_301.pdf-3,what was entergy gulf states louisiana 2019s receivables from the money pool from 2008 to 2011 in millions,7.416150e+04,"add(23596, 63003), add(50131, #0), add(11589, #1), add(#2, const_4), divide(#3, const_2)",5,[table_2]
2,CDW/2015/page_93.pdf-2,"how much more money was expensed per outstanding basic weighted-average share in the year ended dec 31 , 2013 compar...",1.262070e+00,"divide(156.6, 17.3), divide(170.6, 21.9), subtract(#0, #1)",3,"[table_1, text_21, text_9]"
3,GS/2013/page_63.pdf-2,what was the change in millions of total assets from 2012 to 2013?,-2.704800e+04,"subtract(911507, 938555)",1,[table_1]
4,JPM/2008/page_87.pdf-1,"as of december 31 , 2008 , what percentage of authorized repurchase capacity remained under the current stock repurc...",6.200000e-01,"divide(6.2, const_10)",1,"[text_27, text_32]"


## 2. Chunking

Forelesning 4 (lysbilde «Dokumentgrunnlag og chunks er designvalg»):
- *Små chunks gir presise treff; større chunks bevarer mer sammenheng.*
- *Chunk-ID må peke tilbake til originaldokumentet.*

Chunkingen ligger i `src/korpus.py` (`make_chunks`), slik at agenten søker i nøyaktig de samme chunkene. Vi bruker **FinQAs egne ID-er som chunk-ID-er**, så treffene kan sammenlignes direkte med `gold_inds`.

| Del av dokumentet | Chunk | Chunk-ID |
|---|---|---|
| Setning nr. *i* i `pre_text + post_text` | én setning (`CHUNK_STRATEGY = "setning"`) | `text_i` |
| Tabellrad nr. *i* | én rad, skrevet om til tekst med kolonnenavn fra rad 0 | `table_i` |

**Tabellrader blir tekst.** En rad som `['leased facilities2', '2.1', '6.0']` blir
`leased facilities2: unitedstates = 2.1; othercountries = 6.0`.

**To ryddevalg** som må med i rapporten:
1. **Negative tall:** FinQA skriver dem som `-32 ( 32 )`. Vi beholder bare `-32`, fordi kalkulatoren ikke kan lese parentesen.
2. **Tomme setninger** (bare `.`) fjernes. De kan aldri være evidens.

**Kjent svakhet:** tabellrader mister sammenhengen sin. Enheten (`$ in millions`) står i rad 0, og segmentnavnet står ofte i teksten over tabellen. En rad som `sales: 2005 = $ 4625` sier altså ikke *hvilket* segment den gjelder. Dette er feilstedet «meningsenhet kuttes», og det rammer RAG-pipelinen og agenten likt.

In [3]:
# --- Vis chunkene for det første spørsmålet i utvalget ---
# is_gold = True betyr at chunken inneholder evidens fasiten bygger på (gold_inds).
eksempel = utvalg.iloc[0]
chunks_df = pd.DataFrame(make_chunks(examples[eksempel["id"]]))
chunks_df["is_gold"] = chunks_df["source_ids"].map(lambda ids: bool(set(ids) & set(eksempel["gold_inds"])))
print({"id": eksempel["id"], "spørsmål": eksempel["question"], "gold_inds": eksempel["gold_inds"],
       "antall_chunks": len(chunks_df)})
display(chunks_df[chunks_df["is_gold"] | (chunks_df["kind"] == "tabell")].head(8))

# --- Kontroll over hele utvalget ---
# Over så vi på ETT spørsmål. Nå sjekker vi ALLE 50, for å se at chunkingen virker overalt.
# Vi svarer på to spørsmål for hver rapportside:
#   1) Hvor mange chunks ble siden delt i? Det avgjør hvor stor andel TOP_K (5) henter.
#   2) Finnes alle gold_inds blant chunk-ID-ene? Hvis ikke, har chunkingen mistet evidens,
#      og da kan verken RAG-pipelinen eller agenten finne svaret, uansett hvor godt de søker.

# Steg 1: Lag en tom liste. Her samler vi ett resultat per spørsmål.
kontroll = []

# Steg 2: Gå gjennom spørsmålene ett for ett.
# utvalg.iterrows() gir oss én rad om gangen; "_" er radnummeret, som vi ikke trenger.
# q er raden, så q["id"] er spørsmålets ID og q["gold_inds"] er fasit-bitene.
for _, q in utvalg.iterrows():

    # Steg 2a: Del spørsmålets rapportside i chunks (samme funksjon som over).
    chunks = make_chunks(examples[q["id"]])

    # Steg 2b: Samle alle FinQA-ID-ene som finnes blant chunkene, f.eks. {"text_0", "text_1", ..., "table_4"}.
    # Hver chunk har en liste source_ids; vi tar alle ID-ene fra alle chunkene og legger dem i én mengde.
    dekket = {sid for c in chunks for sid in c["source_ids"]}

    # Steg 2c: Finn fasit-biter som MANGLER blant chunkene.
    # set(...) - dekket betyr "det som står i fasitlisten, men ikke blant chunkene".
    # len(...) teller hvor mange det er. 0 betyr at ingenting mangler (det vi håper på).
    mangler = len(set(q["gold_inds"]) - dekket)

    # Steg 2d: Lagre de to svarene for dette spørsmålet.
    kontroll.append({"n_chunks": len(chunks), "gold_mangler": mangler})

# Steg 3: Gjør listen om til en tabell med én rad per spørsmål (kolonnene n_chunks og gold_mangler).
kontroll = pd.DataFrame(kontroll)

# Steg 4: Oppsummer de 50 radene i tre tall og skriv dem ut.
print({
    # Minste, typiske (median) og største antall chunks per rapportside.
    "chunks_per_dokument (min/median/maks)": (int(kontroll.n_chunks.min()), int(kontroll.n_chunks.median()), int(kontroll.n_chunks.max())),
    # Hvor stor del av en typisk side søket henter: TOP_K delt på antall chunks. 0.17 = 17 %.
    "andel_av_dokumentet_top_k_henter (median)": round(float(TOP_K / kontroll.n_chunks.median()), 2),
    # Hvor mange spørsmål som mangler minst én fasit-bit etter chunkingen. Skal være 0.
    "spørsmål_der_gold_mangler_i_chunks": int((kontroll.gold_mangler > 0).sum()),
})

{'id': 'AMT/2016/page_49.pdf-4', 'spørsmål': 'what is the average number of shares per registered holder as of february 17 , 2017?', 'gold_inds': ['text_3'], 'antall_chunks': 20}


,chunk_id,source_ids,kind,text,is_gold
3,text_3,[text_3],tekst,"as of february 17 , 2017 , we had 427195037 outstanding shares of common stock and 153 registered holders .",True
10,table_0,[table_0],tabell,kolonner: 2016 | high | low,False
11,table_1,[table_1],tabell,quarter ended march 31: high = $ 102.93; low = $ 83.07,False
12,table_2,[table_2],tabell,quarter ended june 30: high = 113.63; low = 101.87,False
13,table_3,[table_3],tabell,quarter ended september 30: high = 118.26; low = 107.57,False
14,table_4,[table_4],tabell,quarter ended december 31: high = 118.09; low = 99.72,False
15,table_5,[table_5],tabell,2015: high = high; low = low,False
16,table_6,[table_6],tabell,quarter ended march 31: high = $ 101.88; low = $ 93.21,False


{'chunks_per_dokument (min/median/maks)': (7, 28, 58), 'andel_av_dokumentet_top_k_henter (median)': 0.18, 'spørsmål_der_gold_mangler_i_chunks': 0}


## 3. Indeks: det felles korpuset

Indeksen gjør hver chunk om til en vektor, slik at vi kan måle hvor lik den er spørsmålet. Forelesning 4 og øving 4 skiller mellom to typer retrieval:

| Backend | Type | Hvordan den måler likhet | Styrke | Svakhet |
|---|---|---|---|---|
| `"tfidf"` | **Sparse** | Ord som finnes i både spørsmål og chunk. Sjeldne ord teller mest (IDF). | Treffer presist på tall, år og faguttrykk | *Vocabulary mismatch*: «revenue» finner ikke «sales» |
| `"embedding"` (all-MiniLM-L6-v2, **standard**) | **Dense** | Tette vektorer for meningen i teksten | Bygger bro mellom ulike ordvalg | Mindre presis på eksakte tall og år |

Likheten måles med **cosinuslikhet** i begge tilfeller. Embeddings ble valgt fordi de hentet alle gold-chunks oftere enn TF-IDF på valideringsutvalget (64 % mot 52 %, se `pilot_val.ipynb`).

**Ett korpus, bygget én gang** (`src/korpus.py`, `Korpus`). Embeddingene for alle chunkene i utvalget regnes ut her, og både RAG-pipelinen og agenten søker i det. Søket skjer bare i **spørsmålets egen rapportside** (`RETRIEVAL_SCOPE = "dokument"`).

In [4]:
# --- Bygg det felles korpuset én gang, bare for spørsmålene i utvalget ---
# Korpuset er "biblioteket" RAG-pipelinen søker i: alle rapportsidene, delt i chunks, med en embedding per chunk.
# Det bygges ÉN gang her, og brukes til alle søkene senere. Agenten bygger nøyaktig det samme
# med samme kode (src/korpus.py), så begge søker i det samme biblioteket.

# Steg 1: Start en stoppeklokke, så vi kan se hvor lang tid byggingen tar.
# time.time() gir klokkeslettet akkurat nå, målt i sekunder.
start = time.time()

# Steg 2: Velg ut rapportsidene vi trenger.
# examples har rapportsidene til spørsmålene i utvalget (fra data/finqa_utvalg/).
# {qid: examples[qid] for qid in utvalg["id"]} lager en ordbok i samme rekkefølge som utvalget:
#   for hver spørsmåls-ID (qid) i utvalget  ->  slå opp rapportsiden i examples.
sider = {qid: examples[qid] for qid in utvalg["id"]}

# Steg 3: Bygg korpuset. Korpus(...) gjør to ting for hver rapportside (se src/korpus.py):
#   a) deler siden i chunks (make_chunks, samme som i celle 2)
#   b) gjør hver chunk om til en embedding, en liste med tall som beskriver meningen i teksten
#      (modellen all-MiniLM-L6-v2). Det er disse tallene søket sammenligner med spørsmålet.
# Dette er steget som tar tid, fordi embeddingmodellen må lese alle chunkene.
korpus = Korpus(sider)

# Steg 4: Skriv ut en oppsummering, for å sjekke at alt ble bygget.
print({
    # Hvilken søkemetode som brukes: "embedding" (dense) eller "tfidf" (sparse).
    "backend": korpus.backend,
    # Hvor mange rapportsider korpuset har. Skal være 50, én per spørsmål.
    "dokumenter": len(korpus.chunks),
    # Hvor mange chunks det ble til sammen: legger sammen antall chunks for hver side.
    "chunks_totalt": sum(len(c) for c in korpus.chunks.values()),
    # Hvor lang tid byggingen tok: klokkeslettet nå minus da vi startet.
    "sekunder": round(time.time() - start, 1),
})

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

{'backend': 'embedding', 'dokumenter': 200, 'chunks_totalt': 5601, 'sekunder': 11.2}


## 4. Retrieval (spørretid, steg 1)

RAG-pipelinen søker **én gang**, med spørsmålet selv som query: `korpus.search(spørsmål, doc_id)`. Agenten bruker samme funksjon, men kan søke flere ganger med egne formuleringer.

Forelesning 4 (lysbilde «Hva retrieval-loggen viser") sier at **rang og score** viser hva indeksen prioriterte, mens **menneskelig relevansvurdering** skiller treff fra evidens. FinQA gir oss den relevansvurderingen gjennom `gold_inds`. Derfor legger vi til kolonnen `is_gold`.

**Retrieval-dekning** («Finnes nødvendig evidens i top-k?»):
- `gold_support_retrieved = True` når **alle** `gold_inds` er hentet. Et regnestykke som trenger to tall, kan ikke bli riktig hvis bare det ene er hentet.
- Som tilleggsinformasjon teller vi spørsmål der **minst én** gold-chunk ble hentet. `gold_inds` har ofte det samme tallet to steder, så «alle» er strengt.

Retrieval kaller ikke LLM-en, så dekningen kan måles på hele utvalget gratis.

In [5]:
# --- Retrieval-loggen for eksempelspørsmålet ---
hits = korpus.search(eksempel["question"], doc_id=eksempel["id"])          # top_k = TOP_K
hits_df = pd.DataFrame(hits)
hits_df["is_gold"] = hits_df["source_ids"].map(lambda ids: bool(set(ids) & set(eksempel["gold_inds"])))
alle_gold, noen_gold = gold_coverage(hits, eksempel["gold_inds"])
print({"spørsmål": eksempel["question"], "top_k": TOP_K, "gold_inds": eksempel["gold_inds"],
       "gold_support_retrieved": alle_gold})
display(hits_df[["rank", "score", "chunk_id", "kind", "is_gold", "text"]])

# --- Retrieval-dekning på hele utvalget (ett søk per spørsmål, slik RAG-pipelinen gjør) ---
dekning = pd.DataFrame([
    dict(zip(["alle_gold_hentet", "minst_en_gold_hentet"],
             gold_coverage(korpus.search(q["question"], doc_id=q["id"]), q["gold_inds"])),
         n_gold=len(q["gold_inds"]))
    for _, q in utvalg.iterrows()])
print({"alle_gold_hentet": f"{dekning.alle_gold_hentet.mean():.0%}",
       "minst_en_gold_hentet": f"{dekning.minst_en_gold_hentet.mean():.0%}"})

# Dekning etter hvor mange gold-chunks spørsmålet trenger: flere gold-chunks er vanskeligere.
dekning.groupby("n_gold")[["alle_gold_hentet", "minst_en_gold_hentet"]].mean().round(2).assign(
    antall_spørsmål=dekning.groupby("n_gold").size())

{'spørsmål': 'what is the average number of shares per registered holder as of february 17 , 2017?', 'top_k': 5, 'gold_inds': ['text_3'], 'gold_support_retrieved': True}


,rank,score,chunk_id,kind,is_gold,text
0,1,0.705,text_3,tekst,True,"as of february 17 , 2017 , we had 427195037 outstanding shares of common stock and 153 registered holders ."
1,2,0.566,text_1,tekst,False,"market for registrant 2019s common equity , related stockholder matters and issuer purchases of equity securities th..."
2,3,0.524,text_2,tekst,False,"on february 17 , 2017 , the closing price of our common stock was $ 108.11 per share as reported on the nyse ."
3,4,0.509,text_9,tekst,False,"we have distributed an aggregate of approximately $ 3.2 billion to our common stockholders , including the dividend ..."
4,5,0.486,text_6,tekst,False,"we have two series of preferred stock outstanding , 5.25% ( 5.25 % ) mandatory convertible preferred stock , series ..."


{'alle_gold_hentet': '70%', 'minst_en_gold_hentet': '92%'}


,alle_gold_hentet,minst_en_gold_hentet,antall_spørsmål
n_gold,,,
1,0.95,0.95,94
2,0.57,0.89,83
3,0.20,1.00,15
4,0.12,0.62,8


## 5. Kontekstpakking (spørretid, steg 2)

Forelesning 4: *bare teksten vi pakker her, blir sendt til generatoren.* Feilstedet i dette leddet er at *viktig evidens kuttes eller drukner*, og kontrollgrepet er å **lese ferdig kontekst**.

Hver chunk pakkes som en egen blokk med chunk-ID og type foran. Det er **samme format som agenten får tilbake fra `search`**, så begge systemene ser evidensen presentert likt:

```
[table_3 | tabell]
securities sold under agreements to repurchase1: as of december 2015 = 86069; as of december 2014 = 88215
```

**Rekkefølge:** rangert, med beste treff først, som i øvingen.

**Lengdegrense:** `MAX_CONTEXT_CHARS` stopper pakkingen hvis konteksten blir for lang. Med `TOP_K = 5` korte chunks skjer det nesten aldri, men vi logger det hvis det skjer.

In [6]:
MAX_CONTEXT_CHARS = 4000   # øvre grense for konteksten (omtrent 1000 tokens)

# --- Hovedfunksjon: pakk de hentede chunkene til én kontekst ---
# Returnerer konteksten og chunkene som faktisk kom med.
def pack_context(hits: list, max_chars: int = MAX_CONTEXT_CHARS) -> tuple[str, list]:
    blocks, used_hits, used = [], [], 0
    for h in hits:                                      # rangert rekkefølge: beste treff først
        # Steg 1: Lag en blokk med chunk-ID og type foran teksten (samme format som agentens search).
        block = f"[{h['chunk_id']} | {h['kind']}]\n{h['text']}"
        # Steg 2: Stopp hvis blokken ville gjort konteksten for lang (men ta alltid med minst én).
        if blocks and used + len(block) > max_chars:
            break
        blocks.append(block)
        used_hits.append(h)
        used += len(block)
    # Steg 3: Skill blokkene med en tom linje.
    return "\n\n".join(blocks), used_hits


# --- Les ferdig kontekst for eksempelspørsmålet ---
context, packed = pack_context(hits)
print({"chunks_i_konteksten": [h["chunk_id"] for h in packed], "tegn": len(context)})
print("\n" + context)

# --- Kontroll over hele utvalget: blir noe kuttet, og hvor lang blir konteksten? ---
lengder = []
for _, q in utvalg.iterrows():
    h = korpus.search(q["question"], doc_id=q["id"])
    ctx, used = pack_context(h)
    lengder.append({"tegn": len(ctx), "kuttet": len(used) < len(h)})
lengder = pd.DataFrame(lengder)
print({"tegn (median/maks)": (int(lengder.tegn.median()), int(lengder.tegn.max())),
       "spørsmål_der_chunks_ble_kuttet": int(lengder.kuttet.sum())})

{'chunks_i_konteksten': ['text_3', 'text_1', 'text_2', 'text_9', 'text_6'], 'tegn': 1168}

[text_3 | tekst]
as of february 17 , 2017 , we had 427195037 outstanding shares of common stock and 153 registered holders .

[text_1 | tekst]
market for registrant 2019s common equity , related stockholder matters and issuer purchases of equity securities the following table presents reported quarterly high and low per share sale prices of our common stock on the nyse for the years 2016 and 2015. .

[text_2 | tekst]
on february 17 , 2017 , the closing price of our common stock was $ 108.11 per share as reported on the nyse .

[text_9 | tekst]
we have distributed an aggregate of approximately $ 3.2 billion to our common stockholders , including the dividend paid in january 2017 , primarily subject to taxation as ordinary income. .

[text_6 | tekst]
we have two series of preferred stock outstanding , 5.25% ( 5.25 % ) mandatory convertible preferred stock , series a ( the 201cseries a preferred sto

{'tegn (median/maks)': (838, 2309), 'spørsmål_der_chunks_ble_kuttet': 0}


## 6. Generering av regnestykke (spørretid, steg 3)

Her skiller RAG-pipelinen seg fra en vanlig RAG-kjede. Generatoren skal **ikke** gi svaret. Den skal skrive **ett program** som kalkulatoren regner ut. Da kan vi skille to feiltyper: valgte modellen feil tall eller operasjon (`uttrykk`), eller kunne programmet ikke kjøres (`kalkulator`).

**Prompten ligger i `src/prompts.py`** (`B2_SYSTEM_PROMPT`). Den er bygget av to felles deler:
- `PROGRAM_RULES`: kalkulatorformatet. Bare de fire operasjonene, `#0` for tidligere steg, og *aldri* nøstede operasjoner. Regelen er testet i `pilot_val.ipynb`.
- `NUMBER_RULES`: tallreglene, **ord for ord de samme som i agentens prompt**. Ingen tusenskille, minustegn for negative tall, og andel i stedet for ×100.

Prompten sier også *«Always write a program, even if you are unsure»*, slik agenten har *«Always give a final answer»*. Da kan ingen av dem avstå.

**Ett oppdiktet eksempel** (*one-shot*) viser formatet. Det er ikke fra FinQA, så ingen fasit lekker inn.

**Ett kall, ingen retry.** Svarer modellen i feil format, blir det en `kalkulator`-feil. Å prøve igjen ville gjort RAG-pipelinen til en agent.

**Modellkallet er felles** (`src/llm.py`): samme modell (`gpt-4.1`, temperatur 0), samme API og samme tokentelling som agenten.

In [7]:
# --- Steg 1: Bygg input for ett spørsmål ---
# Først det oppdiktede eksempelet (one-shot), så det ekte spørsmålet med konteksten.
def build_input(question: str, context: str) -> list:
    return [
        {"role": "user", "content": B2_EXAMPLE_USER},
        {"role": "assistant", "content": B2_EXAMPLE_ASSISTANT},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
    ]


# --- Steg 2: Hent selve programmet ut av svaret ---
# Modellen skal svare med bare programmet, men kan legge til ``` eller "Program:".
# Vi tar den første linjen som inneholder op(...). Ellers sendes råteksten videre,
# og da feiler kalkulatoren (det er ønsket: ingen ny sjanse).
PROGRAM_LINE_RE = re.compile(r"(?:add|subtract|multiply|divide)\(")

def extract_program(raw: str) -> str:
    for line in (raw or "").strip().strip("`").splitlines():
        if PROGRAM_LINE_RE.search(line):
            return line.split(":", 1)[-1].strip() if line.lower().startswith("program") else line.strip()
    return (raw or "").strip()


# --- Steg 3: ETT kall til LLM-en, ingen retry ---
def generate_program(question: str, context: str) -> dict:
    response = call_model(build_input(question, context), instructions=B2_SYSTEM_PROMPT)
    raw = response.output_text
    return {"raw_output": raw, "generated_expression": extract_program(raw), **usage(response)}


# --- Vis prompten og det modellen får for eksempelspørsmålet ---
# Selve modellkallet gjøres ikke her. Svarene på hele testutvalget leses inn i seksjon 9,
# og en valgfri live-kjøring finnes i seksjon 12.
print("=== Systemprompt ===\n" + B2_SYSTEM_PROMPT)
print("\n=== Det LLM-en får (siste melding) ===\n" + build_input(eksempel["question"], context)[-1]["content"])

=== Systemprompt ===
You answer numerical questions about financial reports.
You do NOT compute the answer yourself. Instead you write ONE calculator program that computes it.

Program format:
- Steps are separated by ", ". Each step is op(a, b) where op is one of: add, subtract, multiply, divide.
- a and b are numbers, or #0, #1, ... meaning the result of step 0, 1, ...
- Never nest operations inside each other. Use #0, #1, ... instead:
  write add(60.1, 161.9), divide(161.9, #0), not divide(161.9, add(60.1, 161.9)).

Numbers:
- Write numbers without thousands separators and without currency signs: 1234.5, not $1,234.5
- Write negative numbers with a minus sign: -32, not (32)
- Percentages and ratios are answered as fractions in decimal form. Do NOT multiply by 100.
  A change from 50 to 60 is 0.2, not 20. 12.4% is written 0.124.
- Use only numbers that appear in the report.

Always write a program, even if you are unsure.
Reply with the program only, on one line. No explanation.

===

## 7. Beregning og retting (spørretid, steg 4–5)

**Beregning:** programmet fra LLM-en sendes uendret til `run_program` i den felles kalkulatoren. Kaster kalkulatoren feil (ukjent operasjon, et tall den ikke kan lese, deling på null), fanger vi feilen og **logger** den, men prøver ikke på nytt. Svaret blir da `None`, og `failure_stage = "kalkulator"`.

**Retting** med de felles reglene i `src/evaluering.py`, de samme som for agenten:

| Regel | Valg |
|---|---|
| **Hovedregel** (`is_correct`) | Relativ feil < 1 % mot `exe_ans` (avtalt) |
| Prosent | Andel mot andel. Kalkulatoren og `exe_ans` gir begge andeler (0.017, ikke 1.7). |
| Tilleggsmål (`is_correct_rounded`) | Like etter avrunding til 3 desimaler, med desimalaritmetikk slik at like tall alltid rundes likt |
| Skalafeil (`is_scale_error`) | Svaret blir riktig hvis det deles eller ganges med 100. Gir `failure_stage = "formatering"`. |

1 %-regelen er romslig når tallene i teksten er avrundet. «$4.6 billion» i stedet for 4625 fra tabellen kan gi et svar innenfor 1 %. Regelen med 3 desimaler godtar derimot alle svar under 0.0005, også feil svar. Derfor rapporterer vi begge.

In [8]:
# --- Steg 4: Beregning ---
# Kjører programmet med den felles kalkulatoren. Feil fanges og logges, men gir ingen ny sjanse.
def compute(program: str) -> tuple[float | None, str | None]:
    try:
        return run_program(program), None
    except Exception as exc:
        return None, f"{type(exc).__name__}: {exc}"


# --- Kjør steg 4 og 5 på fasitprogrammet til eksempelspørsmålet ---
# (Modellens egne programmer for hele testutvalget leses inn i seksjon 9.)
output, error = compute(eksempel["gold_program"])
print({"program": eksempel["gold_program"], "calculator_output": output,
       "calculator_error": error, "avrundet": format_answer(output),
       "gold_answer": float(eksempel["gold_answer"]), "correct": is_correct(output, eksempel["gold_answer"])})

# --- Kalkulatorfeil i praksis: hva skjer med typiske formatfeil fra en LLM? ---
for program in ["divide(1,234, 5)", "subtract(-32 ( 32 ), 5)", "divide(161.9, add(60.1, 161.9))", "Svaret er 0.017"]:
    print(f"{program:35} -> {compute(program)}")

# --- Reglene på konstruerte tilfeller ---
regeltest = pd.DataFrame([
    {"tilfelle": "riktig, mange desimaler",   "svar": 0.017135, "fasit": 0.01714},
    {"tilfelle": "ganget med 100 (prosent)",  "svar": 1.7135,   "fasit": 0.01714},
    {"tilfelle": "feil tall",                  "svar": 0.0251,   "fasit": 0.01714},
    {"tilfelle": "fortegnsfeil",               "svar": 1.9,      "fasit": -1.9},
    {"tilfelle": "svært lite tall, feil svar", "svar": 0.0002,   "fasit": 0.00001},
    {"tilfelle": "avrundet tall fra teksten",  "svar": 219 / 4600, "fasit": 0.04735},
])
regeltest["correct (1 %)"] = [is_correct(s, f) for s, f in zip(regeltest.svar, regeltest.fasit)]
regeltest["correct_rounded (3 des.)"] = [is_correct_rounded(s, f) for s, f in zip(regeltest.svar, regeltest.fasit)]
regeltest["skalafeil"] = [is_scale_error(s, f) for s, f in zip(regeltest.svar, regeltest.fasit)]
regeltest

{'program': 'divide(427195037, 153)', 'calculator_output': 2792124.424836601, 'calculator_error': None, 'avrundet': 2792124.425, 'gold_answer': 2792124.42484, 'correct': True}
divide(1,234, 5)                    -> (None, 'ValueError: too many values to unpack (expected 2, got 3)')
subtract(-32 ( 32 ), 5)             -> (None, "ValueError: Kan ikke lese tallet: '-32 ( 32'")
divide(161.9, add(60.1, 161.9))     -> (None, "ValueError: Kan ikke lese tallet: 'add(60.1'")
Svaret er 0.017                     -> (None, "ValueError: Fant ingen steg i: 'Svaret er 0.017'")


,tilfelle,svar,fasit,correct (1 %),correct_rounded (3 des.),skalafeil
0,"riktig, mange desimaler",0.017135,0.01714,True,True,False
1,ganget med 100 (prosent),1.713500,0.01714,False,False,True
2,feil tall,0.025100,0.01714,False,False,False
3,fortegnsfeil,1.900000,-1.90000,False,False,False
4,"svært lite tall, feil svar",0.000200,0.00001,False,True,False
5,avrundet tall fra teksten,0.047609,0.04735,True,False,False


## 8. Hele kjeden: `run_b2()`

Nå setter vi steg 1–5 sammen til én funksjon. Den kjører **alltid de samme stegene i samme rekkefølge**, uten løkker og uten retry. Det er dette som gjør RAG-pipelinen til en fast kjede.

```
spørsmål → search → pack_context → generate_program → compute → retting → loggrad
```

**Loggraden har de avtalte feltene** (`src/logg.py`), de samme som agenten. Da kan resultatfilene kobles på `id`:

| Felt | RAG-pipeline | Agenten (til sammenligning) |
|---|---|---|
| `answer` / `raw_answer` | Kalkulatorens resultat / LLM-ens råsvar | Tallet fra `ANSWER:` / sluttsvaret |
| `tool_calls` | Ett `run_program`-kall | Alle `search`- og `calculator`-kall |
| `retrieved_chunks` | De 5 chunkene fra ett søk | Alle chunks fra alle søk |
| `n_model_calls` | Alltid 1 | 1–8 |
| `stopped` | Alltid `svar` | `svar` eller `tvunget_svar` |

**failure_stage** settes av den felles `classify_failure`, i fast rekkefølge: ingen → kalkulator → formatering → retrieval → uttrykk.

**API-feil** (nettverk, kvote) fanges og lagres i `error`. De er ikke feil i RAG-pipelinen og holdes utenfor evalueringen.

Funksjonen kjøres ikke her. Resultatene for hele testutvalget leses inn i seksjon 9. En valgfri live-kjøring på 8 testspørsmål finnes i seksjon 12.

In [9]:
def run_b2(q: pd.Series, top_k: int = TOP_K) -> dict:
    start = time.time()
    gold = float(q["gold_answer"])
    base = dict(condition=CONDITION, split=SPLIT, id=q["id"], question=q["question"], gold_answer=gold,
                gold_program=q["gold_program"], gold_inds=list(q["gold_inds"]), n_steps=int(q["n_steps"]),
                model=MODEL, settings={**settings(), "top_k": top_k})

    # Steg 1: Retrieval i spørsmålets eget dokument (ett søk, spørsmålet som query)
    hits = korpus.search(q["question"], doc_id=q["id"], k=top_k)

    # Steg 2: Kontekstpakking med chunk-ID-er
    context, packed = pack_context(hits)
    all_gold, any_gold = gold_coverage(packed, q["gold_inds"])    # bare det som faktisk ble sendt

    # Steg 3: Ett LLM-kall som gir ett program. API-feil er ikke feil i RAG-pipelinen: de lagres i error.
    try:
        gen = generate_program(q["question"], context)
    except Exception as exc:
        return make_record(**base, error=f"{type(exc).__name__}: {exc}", seconds=round(time.time() - start, 2))

    # Steg 4: Beregning med den felles kalkulatoren
    output, calc_error = compute(gen["generated_expression"])

    # Steg 5: Retting og feilsted med de felles reglene
    correct = is_correct(output, gold)
    return make_record(
        **base,
        answer=output, raw_answer=gen["raw_output"],
        tool_calls=[{"name": "run_program", "arguments": gen["generated_expression"],
                     "output": calc_error if calc_error else output}],
        retrieved_chunks=[{"chunk_id": h["chunk_id"], "source_ids": h["source_ids"], "text": h["text"]} for h in packed],
        n_model_calls=1, n_tool_calls=1, stopped="svar",
        input_tokens=gen["input_tokens"], output_tokens=gen["output_tokens"],
        reasoning_tokens=gen["reasoning_tokens"],
        seconds=round(time.time() - start, 2), error=None,
        correct=correct, correct_rounded=is_correct_rounded(output, gold),
        gold_support_retrieved=all_gold, any_gold_retrieved=any_gold,
        failure_stage=classify_failure(correct, output, gold, all_gold),
    )

## 9. Kjøring på hele utvalget

Nå kjører vi `run_b2` på alle spørsmålene i `SPLIT`. Det er ett API-kall per spørsmål.

**Lagring:** `results/rag_pipeline_{SPLIT}.jsonl`, én JSON-linje per spørsmål med de avtalte feltene. Agenten lagrer til `results/agent_{SPLIT}.jsonl` med de samme feltene.

**Ingen dobbeltbetaling:** finnes filen, leses den inn i stedet for å kjøre på nytt. Sett `RERUN = True` når du har endret noe, for eksempel prompten.

**API-feil** (`error` satt) vises for seg og holdes utenfor evalueringen.

In [10]:
RERUN = False                                          # True = kjør alle spørsmålene på nytt, også med uendret oppsett
RESULTS_PATH = RESULTS_DIR / f"rag_pipeline_{SPLIT}.jsonl"

# --- Steg 0: Gjenbruk en lagret kjøring bare hvis den ble laget med samme oppsett som nå ---
# Er f.eks. modellen byttet i src/config.py, er filen utdatert, og alt kjøres på nytt automatisk.
lagret = read_jsonl(RESULTS_PATH) if RESULTS_PATH.exists() else None
if lagret is not None and not RERUN and not same_settings(lagret, settings()):
    print(f"{RESULTS_PATH.relative_to(ROOT)} ble laget med et annet oppsett enn src/config.py, kjører på nytt")

if lagret is not None and not RERUN and same_settings(lagret, settings()):
    # --- Steg 1a: Les inn tidligere kjøring ---
    resultater_alle = lagret
    print(f"Leste {len(resultater_alle)} resultater fra {RESULTS_PATH.relative_to(ROOT)} (sett RERUN = True for å kjøre på nytt)")
else:
    # --- Steg 1b: Kjør hele kjeden på hvert spørsmål ---
    rader, start = [], time.time()
    for i, (_, q) in enumerate(utvalg.iterrows(), start=1):
        rader.append(run_b2(q))
        if i % 10 == 0:
            print(f"{i}/{len(utvalg)} ferdig, {time.time() - start:.0f} s")
    # --- Steg 2: Lagre som JSONL (avtalt format) ---
    write_jsonl(rader, RESULTS_PATH)
    resultater_alle = pd.DataFrame(rader)
    print(f"Lagret {len(resultater_alle)} resultater til {RESULTS_PATH.relative_to(ROOT)}")

# --- Steg 3: Skill ut API-feil (ikke feil i RAG-pipelinen) ---
api_feil = resultater_alle[resultater_alle["error"].notna()]
resultater = resultater_alle[resultater_alle["error"].isna()].copy()
if len(api_feil):
    display(api_feil[["id", "error"]])

# --- Steg 4: Rask oversikt (grundig evaluering kommer i celle 10) ---
print({"spørsmål": len(resultater_alle), "api_feil": len(api_feil), "riktige": int(resultater["correct"].sum()),
       "tokens_totalt": int(resultater["input_tokens"].sum() + resultater["output_tokens"].sum())})
resultater[["id", "gold_answer", "answer", "correct", "failure_stage"]].head(10)

Leste 200 resultater fra results/rag_pipeline_test.jsonl (sett RERUN = True for å kjøre på nytt)
{'spørsmål': 200, 'api_feil': 0, 'riktige': 121, 'tokens_totalt': 127886}


,id,gold_answer,answer,correct,failure_stage
0,AMT/2016/page_49.pdf-4,2.792124e+06,2.792124e+06,True,ingen
1,ETR/2011/page_301.pdf-3,7.416150e+04,NaN,False,kalkulator
2,CDW/2015/page_93.pdf-2,1.262070e+00,NaN,False,kalkulator
3,GS/2013/page_63.pdf-2,-2.704800e+04,-2.704800e+04,True,ingen
4,JPM/2008/page_87.pdf-1,6.200000e-01,6.200000e-01,True,ingen
5,FIS/2016/page_9.pdf-1,8.008500e-01,8.008475e-01,True,ingen
6,FIS/2017/page_64.pdf-1,-4.138000e-02,-4.138292e-02,True,ingen
7,SYY/2005/page_70.pdf-2,7.190300e-01,7.190291e-01,True,ingen
8,INTC/2013/page_31.pdf-2,-2.200000e-01,1.360000e+00,False,retrieval
9,RSG/2015/page_98.pdf-2,2.147750e+00,NaN,False,kalkulator


## 10. Evaluering

Tabellene lages av den felles `oppsummering` i `src/rapport.py`, så de er regnet ut **nøyaktig likt** for RAG-pipelinen og agenten:

1. **Nøyaktighet** med 95 % Wilson-konfidensintervall, for hovedregelen (1 %) og tilleggsmålet (3 desimaler).
2. **Retrieval recall** mot `gold_inds`: alle hentet, og minst én hentet.
3. **Nøyaktighet med og uten evidens:** er retrieval flaskehalsen?
4. **Ett vs. flere regnesteg** (avtalt inndeling).
5. **Feilsteder.**
6. **Kostnad per spørsmål:** modellkall, verktøykall, tokens og tid.

Med 50 spørsmål (dev) er konfidensintervallene omtrent ±13 prosentpoeng, med 200 (test) omtrent ±7. Dev er til for å justere, test for å konkludere.

In [11]:
tabeller = oppsummering(resultater)
for navn in ["noyaktighet", "recall", "etter_evidens", "etter_steg", "feilsted"]:
    print(f"--- {navn} ---")
    display(tabeller[navn])
tabeller["kostnad"]

--- noyaktighet ---


,mål,andel [95 % KI],antall
0,"riktig, relativ feil < 1 % (hovedregel)","60% [54%, 67%]",121
1,"riktig, 3 desimaler (tilleggsmål)","60% [53%, 67%]",120


--- recall ---


,mål,andel [95 % KI]
0,alle gold_inds hentet,"70% [63%, 76%]"
1,minst én gold_ind hentet,"92% [87%, 95%]"


--- etter_evidens ---


,antall,riktige,andel
gold_support_retrieved,,,
gold mangler,60,13,22%
alle gold hentet,140,108,77%


--- etter_steg ---


,antall,riktige,noyaktighet,alle_gold_hentet
regnesteg,,,,
1 steg,110,64,"58% [49%, 67%]",66%
2+ steg,90,57,"63% [53%, 73%]",74%


--- feilsted ---


,antall,andel
failure_stage,,
ingen,121,60%
retrieval,33,16%
uttrykk,31,16%
kalkulator,14,7%
formatering,1,0%


,RAG-pipeline
modellkall (snitt),1.0
verktøykall (snitt),1.0
input_tokens (snitt),624.0
output_tokens (snitt),15.0
reasoning_tokens (snitt),0.0
sekunder (snitt),4.2
tokens totalt,127886.0


## 11. Failure-logg

Som i øvingen: hver feil får **ett feilsted** og **en neste endring**. Feilstedene følger leddene i kjeden:

| failure_stage | Ledd (forel. 4) | Betyr |
|---|---|---|
| `retrieval` | Retrieval | Nødvendig evidens kom ikke med i konteksten |
| `uttrykk` | Generering | Evidensen var der, men LLM-en lagde feil regnestykke |
| `kalkulator` | Generering → beregning | Programmet kunne ikke kjøres av kalkulatoren |
| `formatering` | Formatering | Riktig tall, men feil skala (×100 eller ÷100) |

**Kontroll av merkelappene.** `retrieval` bygger på `gold_inds`, som ofte har samme tall to steder. Derfor sjekker vi med en uavhengig test: **står alle tallene fra fasitprogrammet i konteksten LLM-en fikk?** (`tall_i_kontekst`). Dette er «les ferdig kontekst» fra forelesningen, gjort automatisk.

**Støy i fasiten.** FinQA er selv inkonsekvent med prosent. Noen fasitprogrammer ganger med 100 (`multiply(#1, const_100)`), andre ikke. Noen `formatering`- og `uttrykk`-feil er derfor egentlig tvetydige fasiter. Det rammer RAG-pipelinen og agenten likt, men bør vurderes manuelt i rapporten.

Loggen lagres som `results/rag_pipeline_{SPLIT}_failure_log.csv`.

In [12]:
logg = failure_log(resultater)
display(failure_oversikt(logg, len(resultater)))
logg.to_csv(RESULTS_DIR / f"rag_pipeline_{SPLIT}_failure_log.csv", index=False)
logg[["failure_stage", "undertype", "tall_i_kontekst", "question", "raw_answer", "gold_program", "answer", "gold_answer"]]

,antall,fasittall_i_kontekst,andel_av_alle_spm
failure_stage,,,
retrieval,33,6,16%
uttrykk,31,30,16%
kalkulator,14,0,7%
formatering,1,1,0%


,failure_stage,undertype,tall_i_kontekst,question,raw_answer,gold_program,answer,gold_answer
43,formatering,,True,what is the five year total return on the goldman sachs group inc.?,"subtract(248.36, 100.00), divide(#0, 100.00)","subtract(248.36, const_100)",1.483600,148.36000
1,kalkulator,annet format,False,what was entergy gulf states louisiana 2019s receivables from the money pool from 2008 to 2011 in millions,No numbers for entergy gulf states louisiana 2019's receivables from the money pool from 2008 to 2011 are provided i...,"add(23596, 63003), add(50131, #0), add(11589, #1), add(#2, const_4), divide(#3, const_2)",NaN,74161.50000
2,kalkulator,avstod / tekst,False,"how much more money was expensed per outstanding basic weighted-average share in the year ended dec 31 , 2013 compar...","Sorry, I do not have enough information to answer this question.","divide(156.6, 17.3), divide(170.6, 21.9), subtract(#0, #1)",NaN,1.26207
9,kalkulator,annet format,False,in december 2015 what was the ratio of the restricted cash and marketable securities to the allowance for doubtful a...,"divide(100.3, [allowance for doubtful accounts amount])","divide(100.3, 46.7)",NaN,2.14775
20,kalkulator,annet format,False,"how is the cash flow of entergy gulf states louisiana affected by the balance from money pool from 2007 to 2008 , in...",No numbers for money pool balances in 2007 or 2008 are provided in the context.,"subtract(55509, 11589)",NaN,43920.00000
...,...,...,...,...,...,...,...,...
176,uttrykk,,True,how bigger were the interest and penalties concerning the interest income in the year 2015?,"subtract(8, 2)","divide(8, 2), subtract(#0, 1)",6.000000,3.00000
187,uttrykk,,True,what was the increase in the settlements with tax authorities as a percent of the tax liabilities observed during 20...,"subtract(-2, -29), divide(#0, -29)","add(29, 4), add(8, 2), divide(29, #0), divide(8, #1), subtract(#2, #3)",-0.931034,0.07879
190,uttrykk,,True,"what are the total off-balance sheet obligations , ( in millions ) ?","add(501, 505), add(#0, 165)","add(505, 165)",1171.000000,670.00000
194,uttrykk,,True,what is the increase in operation and maintenance expenses as a percentage of net revenue in 2003?,"add(7.1, 4.4), divide(#0, 426.6)","add(6.6, 3.7), divide(#0, 426.6)",0.026957,0.02414


## 12. Ekstra: live-demo mot API-et (valgfritt)

Alt over bruker den lagrede testkjøringen, som er grunnlaget for rapporten. Dette steget er et **supplement** for den som vil se RAG-pipelinen i bruk med ekte API-kall.

- Sett `LIVE_DEMO = True` i cellen under og kjør den. Det krever `OPENAI_API_KEY` i `.env`.
- RAG-pipelinen kjøres på de `N_DEMO` = 8 første testspørsmålene, med de samme funksjonene som over.
- Live-svarene vises ved siden av svarene fra den lagrede testkjøringen. Med temperatur 0 blir de stort sett like, men små forskjeller kan forekomme.
- Demoen lagrer ingenting og endrer ikke resultatene over. Den tar under ett minutt og koster omtrent 1 cent.

In [13]:
# --- Valgfritt: sett til True for å kjøre live mot OpenAI (krever OPENAI_API_KEY i .env) ---
LIVE_DEMO = False

from src.config import N_DEMO                     # antall testspørsmål i demoen (8)
from src.llm import har_api_nokkel                # finnes det en API-nøkkel i .env?
from src.rapport import demo_sammenligning        # live-svar ved siden av den lagrede testkjøringen

if not LIVE_DEMO:
    print("Live-demoen er av (LIVE_DEMO = False). Alle resultatene over er fra den lagrede testkjøringen.")
elif not har_api_nokkel():
    print("Fant ingen OPENAI_API_KEY i .env. Legg inn en nøkkel for å kjøre live-demoen.")
else:
    # Steg 1: Kjør systemet live på de N_DEMO første spørsmålene i utvalget.
    demo = pd.DataFrame([run_b2(q) for _, q in utvalg.head(N_DEMO).iterrows()])
    demo["program"] = demo["tool_calls"].map(lambda calls: calls[0]["arguments"] if calls else None)
    # Steg 2: Legg svarene fra den lagrede testkjøringen ved siden av (src/rapport.py).
    sammendrag, demo = demo_sammenligning(demo, RESULTS_DIR / f"rag_pipeline_{SPLIT}.jsonl")
    print(sammendrag)
    display(demo[[c for c in ["id", "gold_answer", "program", "answer", "correct", "svar_testkjøring",
                              "samme_svar", "failure_stage", "input_tokens", "seconds"] if c in demo]])


Live-demoen er av (LIVE_DEMO = False). Alle resultatene over er fra den lagrede testkjøringen.
